# DJ Set Song Recognition (Proof of Concept)

Given a music library and a recorded DJ set exclusively containing songs from that music library, this notebook attempts to identify which songs are playing at each time in the DJ set.

The matter is complicated by the fact that at a given time in my sets I'm usually playing two songs at the same time.

Initial results of this experiment, showing modest success (and a lot of false positives) follow the code:

## Load useful libraries

In [1]:
import librosa

import multiprocessing
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType, ArrayType, StringType

## User settings

In [2]:
output_directory = 'output'

# location of tracks to analyze for building the track feature database
playlist_parquet = '../database/music/output/playlist.parquet'

# location of the recorded DJ set which we want to analyze
filename_show = '/home/emily/Desktop/projects/dj/song_recognition/data/partial-goth-set.mp3'

## Load track file paths from the playlist file

This playlist lists the tracks used in a DJ performance. However, it does not specify the order in which I play them or how I blend tracks together (why we need this software in the first place!).

In [3]:
df_playlist = pd.read_parquet(playlist_parquet)

In [4]:
df_playlist.head(5)

,id,path
0,4822,/media/emily/DJ_Backup/Tracks - A/2025-05-18/7...
1,1443,/media/emily/DJ_Backup/Tracks - A/03 - Can I P...
2,4976,/media/emily/DJ_Backup/Tracks - A/2025-10-25/0...
3,2149,/media/emily/DJ_Backup/Tracks - A/2022-11-07/0...
4,91,/media/emily/DJ_Backup/Tracks - A/Front 242 - ...


In [5]:
df_playlist['path'][0:5].values

array(['/media/emily/DJ_Backup/Tracks - A/2025-05-18/76 - Lunch Box_PN.mp3',
       '/media/emily/DJ_Backup/Tracks - A/03 - Can I Play with Madness (2015 Remaster)_pn.mp3',
       '/media/emily/DJ_Backup/Tracks - A/2025-10-25/01 - Military Fashion Show (Club Hit)_PN.mp3',
       '/media/emily/DJ_Backup/Tracks - A/2022-11-07/01 - Burn_pn.mp3',
       '/media/emily/DJ_Backup/Tracks - A/Front 242 - DJ Hell - Electronicbody-Housemusic CD2 - Headhunter_pn.mp3'],
      dtype=object)

## Define a function that produces a chromagram for a track for a specific octave

By performing this analysis one octave at a time and then appending the results together (later), we obtain a chromagram vector that covers every note on the piano seperately per time step.

In [6]:
def process_octave(y, sr, octave_number, hop_length = 512 * 200, lowest_pitch = 16.35):  # get a more precise number for C0
    chromagram = librosa.feature.chroma_cens(
        y = y,
        sr = sr,
        fmin = lowest_pitch * (octave_number + 1),
        n_octaves = 1,
        hop_length = hop_length,
    )
    return chromagram

## Define function enabling parallelized track analysis

In [7]:
def process_song_from_song_library(filename, track_id):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now, neither efficient nor too costly

    # there is a file not found error I need to debug later
    try:
        y, sr = librosa.load(filename)
    except:
        return None
    
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    results_list = []
    for octave_number in range(0, 9):
        chromagram = process_octave(y_harmonic, sr, octave_number)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    df_all_octaves['id'] = track_id
    return df_all_octaves

## Mathematically process song library to extract features

In [8]:
args_list = []
for i, row in df_playlist.iterrows():
    args_list.append((row['path'], row['id']))

with multiprocessing.Pool(processes = 50) as pool:
    results = pool.starmap(process_song_from_song_library, args_list)

/tmp/ipykernel_1468368/3171468248.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filename)
/home/emily/venvs/ml/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1468368/3171468248.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filename)
/home/emily/venvs/ml/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_1468368/3171468248.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filename)
/home/emily/venvs/ml/lib/python3.10/site-packa

## Assemble song library feature vectors into a DataFrame

In [9]:
df = pd.concat(results, ignore_index = True)

In [10]:
df.head(3)

,C0,C♯0,D0,D♯0,E0,F0,F♯0,G0,G♯0,A0,...,D♯8,E8,F8,F♯8,G8,G♯8,A8,A♯8,B8,id
0,0.344815,0.112089,0.136367,0.172484,0.142443,0.205490,0.174831,0.343460,0.368679,0.270560,...,0.259114,0.191983,0.196706,0.272766,0.181645,0.354552,0.456288,0.247082,0.280853,4822
1,0.344740,0.110633,0.134253,0.167168,0.139185,0.202820,0.173641,0.340443,0.365030,0.277271,...,0.262891,0.193481,0.197752,0.273631,0.182349,0.353875,0.464535,0.242827,0.278242,4822
2,0.344576,0.109610,0.132323,0.162348,0.136196,0.200319,0.172394,0.337547,0.361117,0.283469,...,0.266301,0.195185,0.198989,0.274916,0.183178,0.352304,0.471738,0.238417,0.275931,4822


## Song library QA

To Do:  Need to identify why these rows produce NaNs.

In [11]:
len(df.index)

11325

In [12]:
len(df.dropna().index)

11325

In [14]:
len(df['id'].unique())

185

## Remove rows having NaNs

To Do:  Need to identify why these rows produce NaNs.

In [15]:
df.dropna(inplace = True)

## Save song library vectors for later use

In the long run, we'd prefer using a vector database.

In [16]:
df.to_parquet(output_directory + '/song_library_vectors.parquet')

## Load song library vectors (so we can start from here next time)

In the long run, we'd prefer using a vector database.

In [17]:
pdf = pd.read_parquet(output_directory + '/song_library_vectors.parquet')

## Load recorded show and extract harmonic content

In [18]:
y_show, sr_show = librosa.load(filename_show)
y_show_harmonic, y_show_percussive = librosa.effects.hpss(y_show)

## Define function for extracting per time step features of the recorded DJ set

In [19]:
def process_show(y_harmonic, sr):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now
    results_list = []
    for octave_number in range(0, 9):
        chromagram = process_octave(y_harmonic, sr, octave_number)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    return df_all_octaves

## Compute DJ set features per time step

In [20]:
df_show = process_show(y_show_harmonic, sr_show)

## Recorded show QA

In [21]:
len(df_show.index)

3252

In [22]:
len(df_show.dropna().index)

3252

## Save the show vectors for later use

In [23]:
df_show.to_parquet(output_directory + '/show_vectors.parquet')

## Load the previously saved show vectors

So we can start computation here after leaving it awhile:

In [24]:
df_show = pd.read_parquet(output_directory + '/show_vectors.parquet')

In [25]:
df_show

,C0,C♯0,D0,D♯0,E0,F0,F♯0,G0,G♯0,A0,...,D8,D♯8,E8,F8,F♯8,G8,G♯8,A8,A♯8,B8
0,0.253968,0.134070,0.100559,0.170141,0.078711,0.188392,0.169125,0.278568,0.326929,0.381175,...,0.116411,0.056590,0.165452,0.340058,0.501667,0.201821,0.031296,0.104000,0.191747,0.683396
1,0.257956,0.132972,0.101686,0.164569,0.080406,0.188235,0.169111,0.273880,0.322755,0.380282,...,0.114386,0.061166,0.168905,0.337733,0.506769,0.201279,0.034587,0.103490,0.189390,0.680270
2,0.262335,0.131938,0.102944,0.158705,0.081942,0.188067,0.169228,0.269436,0.319038,0.379379,...,0.113406,0.066155,0.172793,0.335800,0.511332,0.201171,0.037855,0.104126,0.187339,0.676595
3,0.267023,0.130858,0.104200,0.152562,0.083258,0.187923,0.169590,0.265212,0.315755,0.378541,...,0.113301,0.071708,0.177557,0.334321,0.515049,0.201382,0.041269,0.105584,0.185542,0.672533
4,0.271773,0.129649,0.105353,0.146150,0.084311,0.187829,0.170190,0.261190,0.312972,0.377820,...,0.113954,0.077833,0.183537,0.333481,0.517750,0.201738,0.045070,0.107742,0.184091,0.668051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3247,0.328264,0.056888,0.066818,0.072767,0.093677,0.143530,0.185510,0.233534,0.280256,0.388499,...,0.306932,0.103649,0.104618,0.232152,0.466984,0.385055,0.302564,0.235584,0.238357,0.123861
3248,0.323829,0.055678,0.065486,0.070145,0.093704,0.143534,0.184595,0.233118,0.280319,0.391364,...,0.303376,0.099480,0.093658,0.228821,0.478346,0.383913,0.299334,0.233938,0.243031,0.113282
3249,0.319502,0.054572,0.064136,0.067533,0.093732,0.143627,0.183703,0.232813,0.280812,0.394387,...,0.300581,0.095876,0.082888,0.225157,0.489586,0.382440,0.295621,0.231949,0.247500,0.102774
3250,0.315365,0.053314,0.062494,0.064685,0.093708,0.143808,0.182639,0.232464,0.281675,0.397470,...,0.298436,0.092735,0.072583,0.221250,0.500775,0.380491,0.291435,0.229690,0.251790,0.092604


## Start a Spark session

In [26]:
conf = (
    SparkConf()
    .setAppName("MyApp")
    .set("spark.executor.memory", "70G")
    .set("spark.driver.memory", "70G")
    .set("spark.driver.maxResultSize", "70G")
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/17 12:43:06 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/17 12:43:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 12:43:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Save the column names for later use

In [27]:
value_columns = df_show.columns

## Enumerate the time steps

In [28]:
df_show['timestamp'] = df_show.index

## Convert the Pandas dataframes into Spark Dataframes

In [29]:
sdf_show = (
    spark
    .createDataFrame(df_show)
    .orderBy('timestamp')
    .withColumn('array_show', F.array(*value_columns))
    .select('timestamp', 'array_show')
)

In [30]:
sdf_show.show(2)

26/03/17 12:43:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+
|timestamp|          array_show|
+---------+--------------------+
|        0|[0.25396835803985...|
|        1|[0.25795611739158...|
+---------+--------------------+
only showing top 2 rows



In [31]:
sdf_library = (
    spark
    .createDataFrame(pdf)
    .orderBy('id')
    .repartition('id')
    .withColumn('array_library', F.array(*value_columns))
    .select('array_library', 'id')
)

In [32]:
sdf_library.show(2)

+--------------------+----+
|       array_library|  id|
+--------------------+----+
|[0.34639683365821...|2149|
|[0.34315595030784...|2149|
+--------------------+----+
only showing top 2 rows



## Save the show and library vectors in Spark-friendly parquet format

In [33]:
sdf_show.write.mode('overwrite').parquet(output_directory + '/show_vectors_spark.parquet')

In [34]:
sdf_library.write.mode('overwrite').parquet(output_directory + '/library_vectors_spark.parquet')

26/03/17 12:43:24 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## Clear some memory

Not sure whether this is necessary or not...

In [35]:
del(df_show)
del(pdf)

## Load the show and library vectors from the Spark-friendly parquet

So we can start here when the memory is clear:

In [36]:
sdf_show = spark.read.parquet(output_directory + '/show_vectors_spark.parquet')

In [37]:
sdf_library = spark.read.parquet(output_directory + '/library_vectors_spark.parquet')

## Combine the two dataframes with a cross join

Also preemptively repartition it.

In [92]:
sdf_cross_joined = sdf_show.crossJoin(sdf_library)

In [93]:
sdf_cross_joined.show(2)

+---------+--------------------+--------------------+---+
|timestamp|          array_show|       array_library| id|
+---------+--------------------+--------------------+---+
|     1594|[0.25094535946846...|[0.33483257889747...| 91|
|     1595|[0.25330230593681...|[0.33483257889747...| 91|
+---------+--------------------+--------------------+---+
only showing top 2 rows



## Define a user-defined function (UDF) for computing cosine similarity

Computes the cosine similarity between two vectors:

In [94]:
@F.udf(returnType=FloatType())
def compute_cosine_similarity(vector1, vector2):
    cosine_dist = cosine(np.array(vector1), np.array(vector2))
    similarity_score = 1 - cosine_dist
    return float(similarity_score)

## Compute cosine similarity for each row

In [95]:
sdf_cross_joined = (
    sdf_cross_joined
    .withColumn('cosine_similarity', compute_cosine_similarity(F.col('array_show'), F.col('array_library')))
    .drop('array_show', 'array_library')
)

In [96]:
sdf_cross_joined.show(2)

+---------+---+-----------------+
|timestamp| id|cosine_similarity|
+---------+---+-----------------+
|     1594| 91|        0.8350759|
|     1595| 91|        0.8358625|
+---------+---+-----------------+
only showing top 2 rows



## Keep only the top 25th percentile of cosine similarities

This is a simple heuristic for reducing the dataset size.

In [97]:
sdf_cross_joined.repartition(100)

DataFrame[timestamp: bigint, id: bigint, cosine_similarity: float]

In [98]:
print(sdf_cross_joined.count())

36828900


In [100]:
# compute 75th percentile
q75 = sdf_cross_joined.approxQuantile('cosine_similarity', [0.80], 0.05)

In [101]:
q75

[0.8665580749511719]

In [102]:
sdf_cross_joined = sdf_cross_joined.where(F.col('cosine_similarity') >= F.lit(q75[0]))

In [103]:
print(sdf_cross_joined.count())

26/03/18 15:06:37 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#685, array_library#688)#1071 >= 0.8665581) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


7907735


## Aggregate by (timestamp, song_id)

We retain the maximum cosine similarity per (timestamp / song ID) pair:

In [104]:
sdf_cross_joined.repartition('timestamp', 'id')

sdf_agg = (
    sdf_cross_joined
    #.orderBy(F.asc('timestamp'), F.desc('cosine_similarity'))
    .groupBy('timestamp', 'id')
    .agg(
        F.max('cosine_similarity').alias('cosine_similarity'),
    )
)

In [108]:
sdf_agg = sdf_agg.orderBy('timestamp', F.desc('cosine_similarity'))

In [109]:
sdf_agg.show(5)

26/03/18 23:43:04 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#685, array_library#688)#1071 >= 0.8665581) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+
|timestamp|  id|cosine_similarity|
+---------+----+-----------------+
|        0|1176|       0.98020715|
|        0|1280|       0.93996507|
|        0|5241|        0.9312942|
|        0|2173|       0.90333176|
|        0|1937|        0.8942697|
+---------+----+-----------------+
only showing top 5 rows



## Record the rank per timestamp

In [110]:
sdf_agg.repartition('timestamp')

DataFrame[timestamp: bigint, id: bigint, cosine_similarity: float]

In [111]:
window_spec = Window.partitionBy('timestamp').orderBy(F.desc('cosine_similarity'))

sdf_agg_next = (
    sdf_agg
    .orderBy(F.asc('timestamp'), F.desc('cosine_similarity'))
    .withColumn('rank', F.row_number().over(window_spec))
)

In [112]:
sdf_agg_next.show(5)

26/03/18 23:50:39 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#685, array_library#688)#1071 >= 0.8665581) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+----+
|timestamp|  id|cosine_similarity|rank|
+---------+----+-----------------+----+
|        0|1176|       0.98020715|   1|
|        0|1280|       0.93996507|   2|
|        0|5241|        0.9312942|   3|
|        0|2173|       0.90333176|   4|
|        0|1937|        0.8942697|   5|
+---------+----+-----------------+----+
only showing top 5 rows



## Keep the top N per timestamp

In [113]:
n = 3

sdf_agg_next = sdf_agg_next.where(F.col('rank') <= 3)

In [114]:
sdf_agg_next.show(10)

26/03/18 23:56:29 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#685, array_library#688)#1071 >= 0.8665581) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


+---------+----+-----------------+----+
|timestamp|  id|cosine_similarity|rank|
+---------+----+-----------------+----+
|        0|1176|       0.98020715|   1|
|        0|1280|       0.93996507|   2|
|        0|5241|        0.9312942|   3|
|        1|1176|        0.9802379|   1|
|        1|1280|       0.94038284|   2|
|        1|5241|       0.93169487|   3|
|        2|1176|        0.9803439|   1|
|        2|1280|       0.94084287|   2|
|        2|5241|       0.93215847|   3|
|        3|1176|        0.9805271|   1|
+---------+----+-----------------+----+
only showing top 10 rows



## Join to get the track names

In [115]:
df_ranked = sdf_agg_next.toPandas()

26/03/19 00:03:05 WARN ExtractPythonUDFFromJoinCondition: The join condition:(compute_cosine_similarity(array_show#685, array_library#688)#1071 >= 0.8665581) of the join plan contains PythonUDF only, it will be moved out and the join plan will be turned to cross join.


In [120]:
df_mix = (
    pd.merge(
        df_ranked,
        df_playlist,
        how = 'left',
        on = 'id',
    )
    .drop(columns = ['cosine_similarity', 'id'])
)

In [122]:
df_mix['path'] = [x.split('/')[-1].replace('.mp3', '') for x in df_mix['path']]

## Compute time in seconds

In [135]:
# TO FIX: This should be specified in the user settings and propagated through the calculations above
hop_length = 512 * 200
sr = 22050

df_mix['time'] = [x / (sr / hop_length) for x in df_mix['timestamp']]

## Pivot the DataFrame

In [136]:
df_mix_pivoted = df_mix.pivot(index='time', columns='rank', values='path')

## Display estimated results

In [142]:
pd.set_option('display.max_rows', None)
df_mix_pivoted.drop_duplicates().head(200)

rank,1,2,3
time,,,
0.000000,Billy Idol - Cyberpunk - Shock To The System_pn,3851968_Club Wedding_(Jack Rokka Remix)_PN,11 - DSM-V_PN
83.591837,Billy Idol - Cyberpunk - Shock To The System_pn,3851968_Club Wedding_(Jack Rokka Remix)_PN,01 - Epic (Radio Remix Edit)_pn
88.235828,Billy Idol - Cyberpunk - Shock To The System_pn,01 - Epic (Radio Remix Edit)_pn,3851968_Club Wedding_(Jack Rokka Remix)_PN
92.879819,Billy Idol - Cyberpunk - Shock To The System_pn,01 - Epic (Radio Remix Edit)_pn,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN
97.523810,Billy Idol - Cyberpunk - Shock To The System_pn,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN,01 - Epic (Radio Remix Edit)_pn
106.811791,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN,Billy Idol - Cyberpunk - Shock To The System_pn,01 - Epic (Radio Remix Edit)_pn
116.099773,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN,01 - Epic (Radio Remix Edit)_pn,Billy Idol - Cyberpunk - Shock To The System_pn
157.895692,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN,01 - Epic (Radio Remix Edit)_pn,Faith No More - Angel Dust - MidLife Crisis_pn
171.827664,6034307_N.W.O. (Re-Recorded)_(Original Mix)_PN,01 - Epic (Radio Remix Edit)_pn,NØXIA - Pray For Me_pn
